In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\20245179\OneDrive - TU Eindhoven\Research Paper\Data\final_data_cleaned_with_author_names.csv")

In [2]:
df

,parent_id,id,author_id,time,title,text,post_url,reactions,year,nu_words,ha_id,text_ha,author_name
0,ZmVlZGJhY2s6MjA0MDM2MDg0NjAxMzE1NA==,ZmVlZGJhY2s6MjA0MDM2MDg0NjAxMzE1NF8yMDQxMDI4Nj...,100001659249038,2019-04-02T22:14:59,NaN,hoeveel KWh hebben de zonnepanelen op de wezen...,0,0,2019,10,100034431854919,De zonnepanelen op het dak van ons kantoor doe...,WormerWonen
1,ZmVlZGJhY2s6MjA0ODM5MDA5ODU1MTgzMw==,ZmVlZGJhY2s6MjA0ODM5MDA5ODU1MTgzM18yMDc5MTcxNj...,1204819098,2018-12-21T15:37:26,NaN,Dan kom ik langs voor sleutels en krijg er gra...,0,0,2018,17,100031130900343,IN ACTI-UM | Onze vakmannen vervangen bij iede...,Actium
2,ZmVlZGJhY2s6MjA0ODM5MDA5ODU1MTgzMw==,ZmVlZGJhY2s6MjA0ODM5MDA5ODU1MTgzM18yMDc4ODYzNj...,627271250,2018-12-21T10:57:48,NaN,"Mmmm, iets nieuws? Wij hebben in maart geen wi...",0,0,2018,10,100031130900343,IN ACTI-UM | Onze vakmannen vervangen bij iede...,Actium
3,ZmVlZGJhY2s6MjA1MDEwNjkzMTY4ODQzNA==,ZmVlZGJhY2s6MjA1MDEwNjkzMTY4ODQzNF8yMDc1OTI1Mj...,100001478616616,2018-08-01T15:22:31,NaN,Het zou beter zijn als Samenwerking een aantal...,0,0,2018,23,100067754302074,Een hoog energieverbruik is slecht voor het mi...,NaN
4,ZmVlZGJhY2s6MjA1MDEwNjkzMTY4ODQzNA==,ZmVlZGJhY2s6MjA1MDEwNjkzMTY4ODQzNF8yMDczOTE5Nj...,100000855817347,2018-07-31T11:51:36,NaN,Leuk als je in een woning woont met warmtepomp...,0,0,2018,34,100067754302074,Een hoog energieverbruik is slecht voor het mi...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3192,ZmVlZGJhY2s6OTU1MjE0NDg0ODMyNDYw,ZmVlZGJhY2s6OTU1MjE0NDg0ODMyNDYwXzk1NTIyMDUzND...,100001182464975,2019-07-30T10:57:55,NaN,Kijk onze wijk <PERSON> <PERSON> Paulo Mariaa ...,0,2,2019,20,100063528101921,Vorige week zijn de laatste huizen van 't Getf...,De Woonplaats
3193,ZmVlZGJhY2s6OTYwNDYyMjc0MzY0NTIx,ZmVlZGJhY2s6OTYwNDYyMjc0MzY0NTIxXzk2NjA4NTQ4Nz...,596212634,2020-10-22T12:40:07,NaN,Is er genoeg ventilatie en / of kunnen ramen g...,0,0,2020,10,100063452708421,Op de 1e verdieping van het oude schoolgebouw ...,Woongoed Zeist
3194,ZmVlZGJhY2s6OTYwNDYyMjc0MzY0NTIx,ZmVlZGJhY2s6OTYwNDYyMjc0MzY0NTIxXzk2NTQwMTMyNz...,100001224379149,2020-10-21T15:54:40,NaN,<PERSON> kan je dus straks gebruik van maken. ...,0,0,2020,21,100063452708421,Op de 1e verdieping van het oude schoolgebouw ...,Woongoed Zeist
3195,ZmVlZGJhY2s6OTYwNDYyMjc0MzY0NTIx,ZmVlZGJhY2s6OTYwNDYyMjc0MzY0NTIxXzk2MzU0MDgwNz...,100002916564091,2020-10-19T10:14:18,NaN,Komen er nog meer werkplekken bij? Of blijft h...,0,0,2020,16,100063452708421,Op de 1e verdieping van het oude schoolgebouw ...,Woongoed Zeist


In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns


# Group comments by post to capture comment counts per post (sustainability)
grouped = df.groupby("text_ha")["text"].apply(list).reset_index(name="comments")

# Encode posts from Sustainability
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
post_embeddings = model.encode(grouped["text_ha"].tolist(), convert_to_tensor=True, normalize_embeddings=True)
# Encode comments from Other Topics

# Compute real similarities (average per post)
real_avg_similarities = []

comment_cursor = 0

for i, comments in enumerate(grouped["comments"]):
    n_comments = len(comments)
    post_embed = post_embeddings[i].unsqueeze(0)
    
    # Comments are the actual sustainability comments for the post
    comment_embeds = model.encode(comments, convert_to_tensor=True, normalize_embeddings=True)

    sims = util.cos_sim(post_embed, comment_embeds).squeeze().cpu().numpy()
    real_avg_similarities.append(sims.mean())

real_avg_similarities = np.array(real_avg_similarities)

print(f"Real Avg Similarity: {real_avg_similarities.mean():.4f}")

# Random Sampling
num_simulations = 1000
random_avg_similarities = []
all_random_similarities = []

for _ in tqdm(range(num_simulations)):
    sim_list = []

    for i, comments in enumerate(grouped["comments"]):
        n_comments = len(comments)
        post_embed = post_embeddings[i].unsqueeze(0)

        # Randomly sample comments
        random_indices = np.random.choice(len(df["text"]), size=n_comments, replace=False)
        sampled_comments = model.encode(df["text"].iloc[random_indices].tolist(), convert_to_tensor=True, normalize_embeddings=True)

        # Compute similarities
        sims = util.cos_sim(post_embed, sampled_comments).squeeze().cpu().numpy()
        sims = np.atleast_1d(sims)

        # Collect individual similarities
        all_random_similarities.extend(sims)
        sim_list.append(sims.mean())

    random_avg_similarities.append(np.mean(sim_list))

null_distribution = np.array(random_avg_similarities)
from scipy.stats import ttest_ind

# Independent t-test (equal variances)
ttest_t_stat, ttest_p_value = ttest_ind(real_avg_similarities, null_distribution, equal_var=True)

# Print results
print(f"Independent t-test - t-statistic: {ttest_t_stat:.4f}, p-value: {ttest_p_value:.4f}")
# Calculate p-value
p_value = np.mean(null_distribution >= real_avg_similarities.mean())

# Convert to numpy array
all_random_similarities = np.array(all_random_similarities)

# Plot the distribution
plt.figure(figsize=(14, 7))
sns.kdeplot(all_random_similarities, fill=True, color='skyblue', alpha=0.6, label=' Posts & Other Comments')
plt.axvline(all_random_similarities.mean(), color='blue', linestyle='--', label=f'Random Mean: {all_random_similarities.mean():.4f}')

plt.title('Distribution of Random Similarities =(Posts & Other Comments)')
plt.xlabel('Cosine Similarity')
plt.ylabel('Density')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Random Avg Similarity (Posts & Other Comments): {null_distribution.mean():.4f}")
print(f"Empirical p-value: {p_value:.4f}")

import matplotlib.pyplot as plt
import seaborn as sns

# Calculate means
real_mean = real_avg_similarities.mean()
random_mean = all_random_similarities.mean()

# Plot setup
plt.figure(figsize=(14, 7))

# KDE for All Random Similarities
sns.kdeplot(all_random_similarities, fill=True, color='skyblue', alpha=0.6, label='All Random Similarities')
plt.axvline(random_mean, color='blue', linestyle='--', label=f'Random Mean: {random_mean:.4f}')

# KDE for Real Similarities
sns.kdeplot(real_avg_similarities, fill=True, color='orange', alpha=0.6, label='Real Similarities')
plt.axvline(real_mean, color='orange', linestyle='--', label=f'Real Mean: {real_mean:.4f}')

# Plot settings
plt.title('Distribution of Real vs. All Random Similarities')
plt.xlabel('Cosine Similarity')
plt.ylabel('Density (Normalized)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



Couldn't import dot_parser, loading of dot files will not be possible.
Real Avg Similarity: 0.3238


  0%|          | 1/1000 [00:48<13:23:34, 48.26s/it]